In [14]:
import re
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

# 기본 설정
target_url = "https://fanfox.net/manga/d_gray_man/"

options = Options()
# options.add_argument("--headless")
driver = webdriver.Chrome(options=options)

try:
    driver.get(target_url)
    time.sleep(3)

    # 1. 성인 인증 버튼 클릭 (Warning 메시지 처리)
    try:
        confirm_btn = driver.find_element(By.PARTIAL_LINK_TEXT, "Please click here")
        confirm_btn.click()
        print("성인 인증 완료")
        time.sleep(3)
    except:
        pass

    # 2. 모든 챕터 리스트 탐색
    # 제공해주신 HTML 구조: ul.detail-main-list 내의 p.title3 태그 타겟팅
    items = driver.find_elements(By.CSS_SELECTOR, "ul.detail-main-list p.title3")

    raw_data = []
    pattern = re.compile(r"Vol\.(\d+)\s+Ch\.(\d+(\.\d+)?)")

    for item in items:
        # 숨겨진 요소(display:none)까지 포함하여 텍스트 추출
        text = item.get_attribute('textContent').strip()
        
        match = pattern.search(text)
        if match:
            # Vol과 Ch를 모두 정수로 변환 (소수점은 버림 처리)
            vol = int(match.group(1))
            ch = int(float(match.group(2))) 
            raw_data.append((vol, ch))

    # 3. 데이터 가공 (모든 Volume 대상)
    # 볼륨 순으로 정렬
    raw_data.sort()
    
    vol_dict = {}
    for v, c in raw_data:
        if v not in vol_dict:
            vol_dict[v] = []
        vol_dict[v].append(c)

    # 4. 결과 리스트 생성 (페이지 내 모든 Volume 포함)
    vs = []   # Vol 목록
    chs = []  # 각 Vol의 시작 Ch
    che = []  # 각 Vol의 마지막 Ch

    # 추출된 모든 볼륨 번호를 오름차순으로 순회
    for v in sorted(vol_dict.keys()):
        vs.append(v)
        chs.append(min(vol_dict[v]))
        che.append(max(vol_dict[v]))

    # 5. 최종 출력
    print(f"vs = {vs}")
    print(f"chs = {chs}")
    print(f"che = {che}")

except Exception as e:
    print(f"오류 발생: {e}")

finally:
    driver.quit()

vs = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26]
chs = [0, 1, 8, 17, 27, 37, 47, 57, 67, 77, 87, 98, 108, 119, 129, 139, 150, 161, 172, 182, 189, 194, 200, 206, 213, 219, 66]
che = [0, 7, 16, 26, 36, 46, 56, 66, 76, 86, 97, 107, 118, 128, 138, 149, 160, 171, 181, 188, 193, 199, 205, 212, 218, 224, 257]


In [12]:
import os
import time
import random
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import SessionNotCreatedException, WebDriverException
from IPython.display import Audio, display
import numpy as np

cartoon = "Gantz"
front_url = "https://fanfox.net/manga/gantz/v"

vs = [20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37]
chs = [233, 248, 257, 265, 273, 281, 292, 304, 312, 320, 327, 336, 344, 352, 360, 368, 378]
che = [237, 247, 256, 264, 272, 280, 291, 303, 311, 319, 326, 335, 343, 351, 359, 367, 377, 383]

page_start = 1

print('start')

options = Options()
options.add_argument("--headless")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
options.add_argument("--window-size=1920,1080")
options.add_argument("--no-sandbox") # 세션 에러 방지를 위한 옵션 추가
options.add_argument("--disable-dev-shm-usage") # 메모리 부족 에러 방지

os.makedirs(cartoon, exist_ok=True)

for index in range(0, len(vs)):
    vs_str = str(vs[index]).zfill(2)
    
    for chapter in range(chs[index], che[index] + 1):
        chap_str = str(chapter).zfill(3)
        os.makedirs(f"{cartoon}/{chap_str}", exist_ok=True)
        stop_chapter = False        

        for page in range(page_start, 200):
            retry_count = 0
            while retry_count < 5:
                url = f"{front_url}{vs_str}/c{chap_str}/1.html#ipg{page}"
                print(f"\n===== 접속: {url} (시도 {retry_count+1}/5)")

                driver = None
                try:
                    # [수정] 드라이버 생성 자체를 try 블록 안에 배치하여 세션 에러 대응
                    driver = webdriver.Chrome(options=options)
                    driver.get(url)
                    time.sleep(4)

                    # 성인 인증/주의 문구 클릭 프로세스
                    try:
                        continue_btn = driver.find_element(By.PARTIAL_LINK_TEXT, "click here to continue")
                        if continue_btn.is_displayed():
                            print("성인 인증 버튼 감지: 클릭 시도")
                            continue_btn.click()
                            time.sleep(2)
                    except:
                        pass

                    driver.execute_script("window.scrollTo(0, 500);")
                    time.sleep(1)

                    imgs = driver.find_elements(By.CLASS_NAME, "reader-main-img")
                    print("찾은 이미지 개수:", len(imgs))

                    if (len(imgs) == 0) and (page > 3):
                        print("이미지 없음 → 챕터 종료")
                        driver.quit()
                        stop_chapter = True
                        break

                    page_str = "img_" + str(page).zfill(3)
                    gif_detected = False

                    for img in imgs:
                        img_url = img.get_attribute("src")
                        if "loading.gif" in img_url or not img_url:
                            time.sleep(3)
                            img_url = img.get_attribute("src")

                        if img_url.startswith("//"):
                            img_url = "https:" + img_url

                        # 이미지 다운로드 로직
                        cookies = driver.get_cookies()
                        session = requests.Session()
                        for cookie in cookies:
                            session.cookies.set(cookie['name'], cookie['value'])

                        headers = {"User-Agent": "Mozilla/5.0", "Referer": url}
                        img_res = session.get(img_url, headers=headers)
                        img_data = img_res.content
                        filename = img_url.split("/")[-1].split("?")[0]
                        name, ext = os.path.splitext(filename)

                        if ext.lower() == '.gif' and "loading" not in filename:
                            retry_count += 1
                            gif_detected = True
                            driver.quit()
                            break 

                        with open(f"{cartoon}/{chap_str}/{page_str}{ext}", "wb") as f:
                            f.write(img_data)
                        print("다운로드 완료:", filename)

                    if gif_detected:
                        continue 

                    driver.quit()
                    break # 성공 시 while 루프 탈출

                except (SessionNotCreatedException, WebDriverException) as e:
                    print(f"브라우저 세션 에러 발생: {e}")
                    retry_count += 1
                    if driver:
                        try: driver.quit()
                        except: pass
                    time.sleep(5) # 재시도 전 대기

                except Exception as e:
                    print(f"일반 에러 발생: {e}")
                    retry_count += 1
                    if driver:
                        try: driver.quit()
                        except: pass
                    time.sleep(3)

            if stop_chapter:
                break
        print(f"챕터 {chap_str} 완료")

print("전체 완료")

if True:
    fs = 44100
    duration = 0.5

    t = np.linspace(0, duration, int(fs*duration))
    tone = np.sin(2*np.pi*1000*t)

    sound = np.tile(tone, 10)

    display(Audio(sound, rate=fs, autoplay=True))

start

===== 접속: https://fanfox.net/manga/gantz/v20/c233/1.html#ipg1 (시도 1/5)


KeyboardInterrupt: 